In [ ]:
#Solo descomenta esta celda si quieres tu iniciar sesión personal en Google Colab

#from google.colab import drive
#drive.mount('/content/drive')

# Mapas interactivos

In [ ]:
import os
import folium
from pyproj import crs
import geopandas as gpd
import matplotlib.pyplot as plt

Comenzamos creando un mapa básico a través de crear la _instancia_ `Map`

[20.67094003314418, -103.35455606610076]

Si exportamos este mapa sencillo a un `html` podemos verlo de forma local en nuestro navegador.

Genera una nueva carpeta llamada `salidas` dentro de la carpeta `datos`.

In [ ]:
ruta_destino = os.path.join('drive', 'MyDrive', 'tu carpeta del curso', 'datos', 'nueva carpeta: salidas')

El parámetro `tiles` puede cambiarse conforme a la visualización del mapa base que nos interese. Acá puedes checar los [estilos de _tiles_](https://python-visualization.github.io/folium/latest/reference.html) disponibles. **No olvides poner la atribución del proveedor del servicio.**

También de dejo [este enlace](https://leaflet-extras.github.io/leaflet-providers/preview/), donde se incluyen servicios de paga o servicios que solicitan una ``API key`` para ser consumidos.

## Agregar capas al mapa

Ahora, agregamos solamente un `Marker` a nuestro mapa...

Posiblemente no es muy útil tener un solo marcador, la mayoría de las veces tenemos toda una lista de puntos. Veámos cómo integrarla...

In [ ]:
ruta = os.path.join('drive', 'MyDrive', 'tu carpeta del curso', 'datos')

In [ ]:
puntos = gpd.read_file(os.path.join(ruta, 'direcciones-corregidas.kml'), enginee='KML')

Convertimos los puntos a un objeto ``GeoJson``

### Controles de capas

Podemos agregar un objeto `LayerControl` a nuestro mapa, lo cual permite a la persona usuaria tener el control de capas visibles. 

_(Es el cuadrito del lado superior izquierdo)_

En el mapa anterior solamente contamos con una capa (la capa de ubicaciones) pero, vamos a agregar la capa de `Colonias del área metropolitana de Guadalajara` para poder ver cómo funciona el `LayerControl`...

In [ ]:
c = gpd.read_file(os.path.join(ruta, 'colonias.geojson'))

![](../source/images/geojson.png)

In [ ]:
# Mapa base
m = folium.Map(location=[20.67094, -103.35456],
               tiles='cartodbpositron',
               zoom_start=12,
               control_scale=True)

# puntos de lugares recreativos
puntos_gjson = folium.GeoJson(data=puntos,
                              name="Lugares recreativos")
puntos_gjson.add_to(m)

# capa de colonias 
colonias_json = folium.GeoJson(
    data=c,
    name='Colonias',
    style_function=lambda feature: {
        'fillColor': '#ffffcc',
        'color': 'gray',
        'weight': 0.5,
        'fillOpacity': 0.4
    },
    tooltip=folium.GeoJsonTooltip(fields=['MUNICIPIO', 'Tipo', 'POBTOT'])
)

colonias_json.add_to(m)

# control de capas
folium.LayerControl().add_to(m)

# nuestra instancia de mapa
m

**Resumen: ¿Qué hace el `lambda` en `style_function`?**

- `lambda` es una forma rápida de escribir una **función pequeña** en Python.
- En `folium.GeoJson`, se usa para definir **cómo debe verse cada polígono** (por ejemplo, color, opacidad, borde).
- La función **no itera** por sí sola.
- Es **`folium` quien recorre automáticamente cada polígono** y llama a esa función una vez por cada uno.
- Así, **cada colonia del mapa** se dibuja con el estilo que devuelve esa función.

### Plugins

En Folium, los plugins son extensiones que añaden **funcionalidades interactivas avanzadas a los mapas**. Estos plugins están basados en la biblioteca JavaScript ``Leaflet.js``, y permiten integrar herramientas visuales como:

* Heatmap
* MarkerCluster
* FullScreen
* Draw

etc...

In [ ]:
coords = [20.67094, -103.35456]

## `HeatMap`

In [ ]:
from folium.plugins import HeatMap

### Mapa de puntos clusterizados

## `MarkerCluster`

In [ ]:
from folium.plugins import MarkerCluster

### Mapa de coropletas

## `Choropleth`

In [ ]:
c = gpd.read_file(os.path.join(ruta, 'colonias.geojson'))

In [ ]:
# Generar la instancia Map
m = folium.Map(location=[20.67094003314418, -103.35455606610076],
               tiles= 'cartodbpositron',
               zoom_start=12,
               control_scale=True)

folium.Choropleth(
    geo_data=c,
    name='Población en 2020',
    data=c,
    columns=['geoid', 'POBTOT'],
    key_on='feature.id',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    line_color='white', 
    line_weight=0,
    highlight=False, 
    smooth_factor=1.0,
    legend_name= 'Población en el área metropolitana de Guadalajara').add_to(m)

m

Podemos manejar mejor la visualización anterior. Veámos qué podemos hacer para evitar los polígonos en color **negro**.

In [ ]:
import branca.colormap as cm

m = folium.Map(location=[20.67094003314418, -103.35455606610076],
               tiles= 'cartodbpositron',
               zoom_start=12,
               control_scale=True)

colormap = cm.linear.YlOrRd_09.scale(c['POBTOT'].min(), c['POBTOT'].max())

def style_function(feature):
    value = feature['properties']['POBTOT']
    return {
        'fillColor': '#cccccc' if value is None else colormap(value),
        'color': 'white',
        'weight': 0.3,
        'fillOpacity': 0.6
    }

folium.GeoJson(
    c,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['MUNICIPIO', 'NOMCOL1', 'Tipo', 'POBTOT'])
).add_to(m)

colormap.caption = 'Población 2020'
colormap.add_to(m)
m

### Ráster en mapas interactivos

En visualización geoespacial, los datos raster representan información continua en una cuadrícula de celdas, como población, altitud o temperatura. A diferencia de los vectores (puntos, líneas o polígonos), los raster permiten mostrar gradientes o distribuciones espaciales con alta resolución.

En este ejemplo, utilizamos un archivo raster de la base de datos [**GPWv4 (Gridded Population of the World)**](https://daac.ornl.gov/cgi-bin/dsviewer.pl?ds_id=975), el cual contiene estimaciones de población por celda (~1 km²) para el año 1995. Primero, recortamos este archivo global al contorno de México, y luego aplicamos una escala logarítmica para mejorar el contraste visual, ya que los valores de población varían drásticamente entre zonas rurales y urbanas.

Usando ``matplotlib`` y ``folium``, convertimos el raster en una imagen superpuesta ``(ImageOverlay)`` que se ajusta dinámicamente al mapa base. Finalmente, añadimos una barra de colores (colormap) para interpretar la intensidad poblacional por celda.

In [ ]:
#!pip install rasterio

In [ ]:
import rasterio

In [ ]:
with rasterio.open(os.path.join(ruta, "poblacion_mexico_1995.tif")) as src:
    data = src.read(1)
    plt.imshow(data, cmap="OrRd")
    plt.title("GPWv4 - Población en México (1995)")
    plt.colorbar(label="Población estimada")
    plt.axis("off")
    plt.show()

In [ ]:
import numpy as np

with rasterio.open(os.path.join(ruta, "poblacion_mexico_1995.tif")) as src:
    data = src.read(1).astype(float)

data[data < 0] = np.nan

# calculamos escala logarítmica
log_data = np.log1p(data)  

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# escala original
im1 = axes[0].imshow(data, cmap="OrRd")
axes[0].set_title("Escala original (población total)")
axes[0].axis("off") #oculta los ejes del gráfico
fig.colorbar(im1, ax=axes[0], shrink=0.8, label="Población")

# escala logarítmica
im2 = axes[1].imshow(log_data, cmap="OrRd")
axes[1].set_title("Escala logarítmica (log1p)")
axes[1].axis("off")
fig.colorbar(im2, ax=axes[1], shrink=0.8, label="log(1 + población)")

plt.tight_layout()
plt.show()

Al transformar los datos a escala logarítmica, reducimos la influencia de los valores extremadamente altos y ampliamos las diferencias entre los valores bajos. Esto es útil cuando los datos están **muy desbalanceados o sesgados**, como ocurre con la población: unas pocas celdas tienen millones de personas, mientras que la mayoría tienen muy pocas. Aplicar ``log(1 + x)`` permite visualizar mejor los patrones generales sin que los valores extremos dominen la gráfica o el mapa.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# escala origina
axes[0].hist(data[~np.isnan(data)].flatten(), bins=100, color="steelblue", edgecolor="white")
axes[0].set_title("Histograma - Población original")
axes[0].set_xlabel("Población estimada por celda")
axes[0].set_ylabel("Frecuencia")
axes[0].set_yscale("log")  

# escala logarítmica
axes[1].hist(log_data[~np.isnan(log_data)].flatten(), bins=100, color="purple", edgecolor="white")
axes[1].set_title("Histograma - Escala logarítmica")
axes[1].set_xlabel("log(1 + población)")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

In [ ]:
np.sum(data == 0)  # número de celdas con población 0

Este enfoque permite integrar información compleja y continua en mapas interactivos, haciéndola más accesible y visualmente clara para cualquier persona usuaria.

In [ ]:
from matplotlib import cm
from PIL import Image
from io import BytesIO
import base64
import branca.colormap as bcm

In [ ]:
# cargar el ráster 
with rasterio.open(os.path.join(ruta, "poblacion_mexico_1995.tif")) as src:
    data = src.read(1).astype(float)   # convertimos a float para permitir np.nan
    bounds = src.bounds

# manejar NoData
data[data < 0] = np.nan #SI CAMBIAMOS --> data[(data <= 0)] = np.nan

# aplicar escala logarítmica
log_data = np.log1p(data)

# creamos colormap 
colormap = bcm.linear.inferno.scale(np.nanmin(data), np.nanmax(data))
colormap = colormap.to_step(index=[0, 100, 1000, 10000, 50000, 100000, 500000, 1_000_000])
colormap.caption = 'Población estimada por celda (1995)'

# normalizamos valores entre 0 y 1
norm_data = (log_data - np.nanmin(log_data)) / (np.nanmax(log_data) - np.nanmin(log_data))

# aplicamos colormap
rgba = cm.inferno(norm_data)

# convertimos a imagen (escala RGB 0–255)
img = (rgba[:, :, :3] * 255).astype(np.uint8)
image = Image.fromarray(img)

# codificamos la imagen como base64 (para Folium)
buffer = BytesIO()
image.save(buffer, format="PNG")
img_b64 = base64.b64encode(buffer.getvalue()).decode()

# coordenadas para la superposición
image_bounds = [[bounds.bottom, bounds.left], [bounds.top, bounds.right]]

# mapa base
m = folium.Map(location=[23.6345, -102.5528],
               zoom_start=5,
               tiles='cartodbpositron')

# agregamos la imagen como capa
folium.raster_layers.ImageOverlay(
    image=f"data:image/png;base64,{img_b64}",
    bounds=image_bounds,
    opacity=0.7,
    name="Población (1995, escala log)",
).add_to(m)

# integramos todo :)
colormap.add_to(m)
folium.LayerControl().add_to(m)

m